## 1. Environment Setup
Clone the AquaMind repo from GitHub and install all required dependencies.

In [2]:
# ── Cell 0 — Config ───────────────────────────────────────────────────────────
# Central config for the entire notebook.
# Change values here — all cells below will use them automatically.

REPO_PATH    = '/kaggle/working/AquaMind'                    # where the repo will be cloned
EPOCHS       = 50                                             # number of training epochs
IMGSZ        = 640                                            # input image size for YOLOv8
RUN_NAME     = 'run_an_3r_4r_5r_8c_2026_06_20_2121'                  # name of this training run

print(f"Config loaded: REPO_PATH={REPO_PATH}, EPOCHS={EPOCHS}, IMGSZ={IMGSZ}, RUN_NAME={RUN_NAME}")


Config loaded: REPO_PATH=/kaggle/working/AquaMind, EPOCHS=50, IMGSZ=640, RUN_NAME=run_an_3r_4r_5r_8c_2026_06_20_2121


In [3]:
# ── Cell 1 — git clone ───────────────────────────────────────────────────────────

%cd /kaggle/working
!rm -rf AquaMind
!git clone https://github.com/govinda-lienart/AquaMind.git
%cd /kaggle/working/AquaMind
print()
print("GitHub repo cloned successfully.")

/kaggle/working
Cloning into 'AquaMind'...
remote: Enumerating objects: 788, done.
remote: Counting objects: 100% (193/193), done.
remote: Compressing objects: 100% (173/173), done.
remote: Total 788 (delta 40), reused 171 (delta 19), pack-reused 595 (from 1)
Receiving objects: 100% (788/788), 3.78 MiB | 33.96 MiB/s, done.
Resolving deltas: 100% (369/369), done.
/kaggle/working/AquaMind

GitHub repo cloned successfully.


In [10]:
# ── Cell 2 — Install dependencies ────────────────────────────────────────────
# ultralytics  → YOLOv8 training and inference
# dvc          → data version control (tracks dataset versions like git tracks code)
# dvc-gdrive   → DVC plugin to use Google Drive as the remote storage backend

!pip install ultralytics dvc dvc-gdrive -q

import ultralytics, dvc
print(f"ultralytics: {ultralytics.__version__}")
print(f"dvc:         {dvc.__version__}\n")
print("All dependencies installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.1/470.1 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.3/79.3 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 451.2/451.2 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.2/214.2 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.2/74.2 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 40.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37

## 2. Data Pipeline
Load the dataset from Google Drive via DVC, then verify the structure is correct before training.

In [5]:
# ── Cell 3 — Inspect dataset.yaml ────────────────────────────────────────────
# dataset.yaml tells YOLOv8 where to find the data and what classes exist.
# path   → root folder of the dataset (relative to this repo)
# train  → subfolder containing training images
# val    → subfolder containing validation images
# nc     → number of classes
# names  → class labels (index 0 = danio_rerio, index 1 = reflection)

with open('/kaggle/working/AquaMind/dataset.yaml', 'r') as f:
    print(f.read())

path: dataset/dataset_ann_3r_4r_5r_8c_2026_06_30_2111
train: images/train
val: images/val
nc: 2
names:
- danio_rerio
- reflection



In [6]:
# ── Cell 4 — Load service account google drive + save credentials ─────────────────────────
from kaggle_secrets import UserSecretsClient
import os

secret = UserSecretsClient().get_secret("GDRIVE_SA_KEY")
os.makedirs('/kaggle/working/AquaMind/secrets', exist_ok=True)

creds_path = '/kaggle/working/AquaMind/secrets/aquamind-497014-5b34149af3fb.json'
with open(creds_path, 'w') as f:
    f.write(secret)

if os.path.exists(creds_path) and os.path.getsize(creds_path) > 0:
    print(f"Service account saved. ({os.path.getsize(creds_path)} bytes)")
else:
    print("ERROR: credentials file missing or empty!")

Service account saved. (2362 bytes)


In [7]:
# ── Cell 5 — Inspect dataset.dvc ────────────────────────────────────────────
# dataset.dvc is the DVC pointer file committed to git.
# It contains the md5 hash of the dataset — DVC uses this hash to fetch
# the correct version of the data from Google Drive.
# If this hash is old, DVC will pull the wrong dataset.

with open('/kaggle/working/AquaMind/dataset.dvc', 'r') as f:
    print(f.read())

outs:
- md5: 2df7597754ddfca1592e924bec4ce086.dir
  size: 631514010
  nfiles: 1805
  hash: md5
  path: dataset



In [12]:
# ── Cell 6 — DVC pull ───────────────────────────────────────────────────────
# DVC reads dataset.dvc, finds the md5 hash, and downloads the matching
# dataset version from Google Drive into the dataset/ folder.
# -q flag suppresses verbose output — prints only when complete.

!dvc pull -q
print("DVC pull complete.")

DVC pull complete.


In [13]:
# ── Cell 7 — Verify dataset ───────────────────────────────────────────────────
import yaml, os

with open(f'{REPO_PATH}/dataset.yaml') as f:
    cfg = yaml.safe_load(f)

base = f"{REPO_PATH}/{cfg['path']}"
print(f"Dataset: {cfg['path']}")
print(f"Train:   {len(os.listdir(base + '/images/train'))} images")
print(f"Val:     {len(os.listdir(base + '/images/val'))} images")


Dataset: dataset/dataset_ann_3r_4r_5r_8c_2026_06_30_2111
Train:   304 images
Val:     77 images


## 3. Training
Fine-tune YOLOv8s on the AquaMind zebrafish dataset.

In [14]:
# ── Cell 8 — Train YOLOv8 ────────────────────────────────────────────────────
# yolov8s.pt  → pretrained nano YOLOv8 model (good balance speed/accuracy)
# data        → path to dataset.yaml (class names + train/val paths)
# epochs      → number of full passes through the training data
# imgsz       → all images resized to 640x640 before being fed to the model
# project     → output folder for results (weights, metrics, plots)
# name        → subfolder name for this specific run

!yolo detect train \
  data={REPO_PATH}/dataset.yaml \
  model=yolov8s.pt \
  epochs={EPOCHS} \
  imgsz={IMGSZ} \
  project=/kaggle/working/aquamind_yolov8 \
  name={RUN_NAME}

import os
weights_path = f'/kaggle/working/aquamind_yolov8/{RUN_NAME}/weights/best.pt'
if os.path.exists(weights_path):
    print(f'Training complete. Best weights saved to {weights_path}')
else:
    print('WARNING: Training may have failed — best.pt not found.')

Ultralytics 8.4.83 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/AquaMind/dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=run_an_3r_4r_5r_8c_2026_06_20_2121, nbs=64, nms=False, opset=None, optimize=False, optimi

## 4. Export Results
Zip the training output and download it from the Output panel.

In [15]:
# ── Cell 9 — Zip results ──────────────────────────────────────────────────────
# Packages the full run output folder into a zip file.
# Contains: best.pt (best weights), last.pt, metrics, confusion matrix, plots.
# Download it from the Kaggle Output panel on the right.

import shutil
from datetime import datetime

date_str = datetime.now().strftime('%Y%m%d_%H%M')
run_dir  = f'/kaggle/working/aquamind_yolov8/{RUN_NAME}'
zip_name = f'/kaggle/working/aquamind_yolov8_{RUN_NAME}_{date_str}'

shutil.make_archive(zip_name, 'zip', run_dir)
print(f'Zip ready: {zip_name}.zip')
print('Download it from the Output panel on the right.')

Zip ready: /kaggle/working/aquamind_yolov8_run_an_3r_4r_5r_8c_2026_06_20_2121_20260630_1434.zip
Download it from the Output panel on the right.
